<img src="../../../../On-Site-Round/Help_BOBAI/Solution/figs/IOAI-Logo.png" alt="IOAI Logo" width="200" height="auto">

[IOAI 2024 (Burgas, Bulgarie), epreuve sur site](https://ioai-official.org/bulgaria-2024)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SKonteye/IOAI-2024/blob/main/fr/On-Site-Round/Help_BOBAI/Solution/Help_BOBAI_Solution.ipynb)

# Help BOBAI : solution de reference

## Jeu de donnees d'entrainement

In [ ]:
import torch

dataset = torch.load('../../../../On-Site-Round/Help_BOBAI/training_set/train-dev_dataset_with_labels.pt')

inputs = dataset[:,:,:-1]
labels = dataset[:, :, -1]


In [ ]:
inputs.shape

In [ ]:
# prompt : compter le nombre d'etiquettes differentes et les occurrences de chacune

import torch

unique_labels, counts = torch.unique(labels, return_counts=True)
num_labels = unique_labels.numel()

print("Number of different labels:", num_labels)
print("Count of each label:")
for label, count in zip(unique_labels, counts):
    print(f"Label {label}: {count}")


In [ ]:
# prompt : separer train/test avec les entrees et les etiquettes

from sklearn.model_selection import train_test_split

train_inputs, test_inputs, train_labels, test_labels = train_test_split(
    inputs, labels, test_size=0.2, random_state=42
)

# Solution

Vous trouverez ci-dessous une solution de reference tres naive : pour un vecteur d'entree donne, soit nous attribuons aleatoirement l'une des nouvelles etiquettes (5 et 6) avec une probabilite uniforme dans une classification a 7 classes, soit nous utilisons le classifieur de base pour produire une prediction.

Vous pouvez remplacer le code ci-dessous par votre propre solution.

In [ ]:
# prompt : placer les elements dont l'etiquette est superieure a 4 dans un autre tableau et entrainer un KNN dessus

# Recuperer les indices des echantillons dont les etiquettes sont superieures a 4
indices_greater_than_4 = (train_labels > 4).nonzero()

# Extraire les entrees et etiquettes correspondantes
train_inputs_greater_than_4 = train_inputs[indices_greater_than_4[:, 0], indices_greater_than_4[:, 1]]
train_labels_greater_than_4 = train_labels[indices_greater_than_4[:, 0], indices_greater_than_4[:, 1]] - 5  # Ajuster les etiquettes pour commencer a 0

In [ ]:
train_inputs_greater_than_4.shape, train_inputs.shape

In [ ]:
# prompt : creer d'abord de nouvelles etiquettes ou les etiquettes inferieures a 5 valent 0, celles egales a 5 valent 1 et celles egales a 6 valent 2
# puis construire un KNN a partir de ces nouvelles etiquettes
# et penser a utiliser toutes ces entrees et etiquettes provenant de train

# Creer de nouvelles etiquettes
new_train_labels = torch.where(train_labels < 5, 0, torch.where(train_labels == 5, 1, 2))

In [ ]:
# prompt : utiliser StandardScaler de sklearn pour normaliser train_input_reshape

import torch
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler

# Remodeler les donnees d'entree pour le KNN
train_inputs_reshaped = train_inputs.reshape(train_inputs.shape[0] * train_inputs.shape[1], train_inputs.shape[2])
new_train_labels_reshaped = new_train_labels.reshape(-1)

# Normaliser les donnees d'entree remodelees
#scaler = StandardScaler()
#train_inputs_reshaped_normalized = scaler.fit_transform(train_inputs_reshaped)

# Entrainer un classifieur KNN sur les nouvelles etiquettes normalisees
knn = KNeighborsClassifier(n_neighbors=3, weights='distance')  # Ajuster n_neighbors si necessaire
knn.fit(train_inputs_reshaped, new_train_labels_reshaped)

In [ ]:
train_inputs_reshaped.shape, train_inputs_reshaped.shape

In [ ]:
train_inputs_reshaped = torch.tensor(train_inputs_reshaped, dtype=torch.float)
train_inputs_reshaped = train_inputs_reshaped.unsqueeze(1)

In [ ]:
train_inputs_reshaped.shape

In [ ]:
import torch
import random

class SevenWayClassifier():
  def __init__(self, ):
    base_clf = torch.nn.Linear(in_features=768, out_features=5, bias=True)
    base_clf.load_state_dict(torch.load("../../../../On-Site-Round/Help_BOBAI/training_set/base_classifier.pth"))
    self.base_clf = base_clf

  def base_classification(self, input_vector):

    with torch.no_grad():
      logits = self.base_clf(input_vector)
      preds = torch.softmax(logits, 1)
      predicted_class = preds.argmax(dim=1).numpy()[0]

    return predicted_class

  def get_preds(self, input_vector):
    with torch.no_grad():
      logits = self.base_clf(input_vector)
      preds = torch.softmax(logits, 1)
    return preds

  def __call__(self, input_vector):
    new_train_labels_reshaped.shape, train_inputs_reshaped.shape
    which = knn.predict(input_vector)[0]
    if (which):
      return which + 4
    return self.base_classification(input_vector)

clf = SevenWayClassifier()

## Inference et evaluation

In [ ]:
from sklearn.metrics import f1_score

def compute_f1(labels, predictions):
  return f1_score(labels, predictions, average='macro')

In [ ]:
from tqdm import tqdm

def inference(clf, input_vectors):
  predictions = []
  input_vectors = input_vectors.reshape(input_vectors.shape[0] * input_vectors.shape[1], input_vectors.shape[2])
  input_vectors = torch.tensor(input_vectors, dtype=torch.float)
  input_vectors = input_vectors.unsqueeze(1)
  for sample in tqdm(input_vectors):
    predictions.append(clf(sample))
  return predictions

In [ ]:
train_inputs.shape, train_inputs_reshaped.shape

In [ ]:
predictions = inference(clf, test_inputs)

f1 = compute_f1(predictions, test_labels)
print('\nNaive solution F1', f1)

## Jeu de donnees de validation

In [ ]:
# Le leaderboard peut fonctionner ou non... S'il ne fonctionne pas, soyez indulgents. Nous essaierons de le remettre en route.

import pandas as pd
import numpy as np

# 30 % des donnees de test
def submission_to_csv(pred: np.ndarray, output_fpath: str = "submission.csv"):
    pred = np.array(pred).flatten()
    data_size = pred.size
    df = pd.DataFrame({
        "ID": np.arange(data_size),
        "class": pred
    })

    df.to_csv(output_fpath, index=False)

eval_inputs = torch.load('../../../../On-Site-Round/Help_BOBAI/Solution/validation_set/eval_dataset.pt')

eval_predictions = inference(clf, eval_inputs)

submission_to_csv(eval_predictions)

## Jeu de donnees de test

In [ ]:
# NE MODIFIEZ PAS CETTE CELLULE
test_inputs = torch.load('../../../../On-Site-Round/Help_BOBAI/Solution/test_set/test_dataset.pt')

split='test'

test_predictions = inference(clf, test_inputs)

with open('{}_predictions.txt'.format("Team Name"), 'w') as outfile:
  outfile.write('\n'.join([str(p) for p in test_predictions]))